In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, 
                              confusion_matrix, roc_auc_score, classification_report)
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

os.makedirs('figures', exist_ok=True)

# Panel'i yükle
panel = pd.read_parquet('data/panel.parquet')
panel['year_month'] = pd.PeriodIndex(panel['year_month'], freq='M')
print(f"Panel: {panel.shape}")
print(panel.head())
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier


In [ ]:
# Özellik grupları
WEATHER_FEATURES = ['sunshine_hours','daylight_hours','temperature_2m_mean',
                    'temperature_2m_max','temperature_2m_min',
                    'precipitation_sum','rain_sum','precipitation_hours']
CONTROL_FEATURES = ['unemployment_rate','inflation_yoy','covid_dummy',
                    'month_sin','month_cos']
TARGETS = ['CCI','retail_index']

# Türkiye enflasyonu çok yüksek — log transform
panel['inflation_log'] = np.log1p(panel['inflation_yoy'].clip(lower=0))

# Eksik değerleri ülke ortalamasıyla doldur (basit imputation)
for col in WEATHER_FEATURES + CONTROL_FEATURES + TARGETS:
    if col in panel.columns:
        panel[col] = panel.groupby('country')[col].transform(
            lambda x: x.fillna(x.mean()))

# Geriye kalan NaN satırlarını düşür (özellikle UK retail eksik)

print(f"NaN düşürmeden önce: {panel.shape}")
panel_clean = panel.dropna(subset=WEATHER_FEATURES + ['CCI'])
print(f"NaN düşürdükten sonra (CCI için): {panel_clean.shape}")

# Retail için ayrı bir alt-set (UK çoğunlukla NaN)
panel_retail = panel.dropna(subset=WEATHER_FEATURES + ['retail_index'])
print(f"Retail için: {panel_retail.shape}")


In [ ]:
fig, ax = plt.subplots(figsize=(12,9))
corr_features = WEATHER_FEATURES + ['unemployment_rate','inflation_yoy','CCI','retail_index']
sns.heatmap(panel_clean[corr_features].corr(), annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, ax=ax, cbar_kws={'label':'Pearson r'})
ax.set_title('Tüm Değişkenler Arası Korelasyon', fontsize=13)
plt.tight_layout()
plt.savefig('figures/ml_01_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
scaler = StandardScaler()
X_weather = scaler.fit_transform(panel_clean[WEATHER_FEATURES])

pca = PCA(n_components=8)
X_pca_full = pca.fit_transform(X_weather)

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1,9), pca.explained_variance_ratio_, color='steelblue')
axes[0].plot(range(1,9), np.cumsum(pca.explained_variance_ratio_), 'ro-')
axes[0].set_xlabel('Bileşen')
axes[0].set_ylabel('Varyans oranı')
axes[0].set_title('Scree Plot — Hava Değişkenleri PCA')
axes[0].grid(alpha=0.3)

# Loading'ler
loadings = pd.DataFrame(pca.components_[:2].T, 
                        columns=['PC1','PC2'], index=WEATHER_FEATURES)
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('PC1 & PC2 Loadings')
plt.tight_layout()
plt.savefig('figures/ml_02_pca_weather.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"PC1 + PC2 toplam varyansın {pca.explained_variance_ratio_[:2].sum()*100:.1f}%'ini açıklıyor")

# panel_clean'e PC sütunlarını ekle
panel_clean = panel_clean.copy()
panel_clean['weather_PC1'] = X_pca_full[:, 0]
panel_clean['weather_PC2'] = X_pca_full[:, 1]

# panel_retail'e de aynı PCA dönüşümünü uygula
panel_retail = panel_retail.copy()
X_pca_retail = pca.transform(scaler.transform(panel_retail[WEATHER_FEATURES]))
panel_retail['weather_PC1'] = X_pca_retail[:, 0]
panel_retail['weather_PC2'] = X_pca_retail[:, 1]

print("✅ panel_clean ve panel_retail'e weather_PC1, weather_PC2 eklendi")
print(f"   panel_clean shape: {panel_clean.shape}")
print(f"   panel_retail shape: {panel_retail.shape}")


In [ ]:
features_for_lr = ['weather_PC1','weather_PC2','unemployment_rate',
                    'inflation_log','covid_dummy','month_sin','month_cos']
X = panel_clean[features_for_lr].values
y = panel_clean['CCI'].values

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr = LinearRegression()
cv_rmse = -cross_val_score(lr, X, y, cv=kf, scoring='neg_root_mean_squared_error')
cv_r2 = cross_val_score(lr, X, y, cv=kf, scoring='r2')

print(f"=== Linear Regression — CCI ===")
print(f"CV RMSE: {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}")
print(f"CV R²:   {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

# Tüm veriyle fit + katsayılar
lr.fit(X, y)
coef_df = pd.DataFrame({
    'feature': features_for_lr,
    'coefficient': lr.coef_,
    'abs_coef': np.abs(lr.coef_)
}).sort_values('abs_coef', ascending=False)
print("\nKatsayılar (büyüklük sırasına göre):")
print(coef_df)

# Görselleştir
fig, ax = plt.subplots(figsize=(10,5))
colors = ['steelblue' if c > 0 else 'salmon' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title(f'CCI Linear Regression Katsayıları (R²={cv_r2.mean():.2f})')
plt.tight_layout()

plt.savefig('figures/ml_03_lr_coefficients_cci.png', dpi=150)
plt.show()


In [ ]:
X_r = panel_retail[features_for_lr].values
y_r = panel_retail['retail_index'].values

lr_r = LinearRegression()
cv_rmse_r = -cross_val_score(lr_r, X_r, y_r, cv=kf, scoring='neg_root_mean_squared_error')
cv_r2_r = cross_val_score(lr_r, X_r, y_r, cv=kf, scoring='r2')

print(f"=== Linear Regression — Retail Index ===")
print(f"CV RMSE: {cv_rmse_r.mean():.3f} ± {cv_rmse_r.std():.3f}")
print(f"CV R²:   {cv_r2_r.mean():.3f} ± {cv_r2_r.std():.3f}")

lr_r.fit(X_r, y_r)
coef_df_r = pd.DataFrame({
    'feature': features_for_lr,
    'coefficient': lr_r.coef_
}).sort_values('coefficient', key=abs, ascending=False)
print("\nKatsayılar:")
print(coef_df_r)


In [ ]:
# Binary target: bir sonraki ayın CCI'sı bugünden yüksek mi?
panel_clean = panel_clean.sort_values(['country','year_month']).reset_index(drop=True)
panel_clean['CCI_next'] = panel_clean.groupby('country')['CCI'].shift(-1)
panel_clean['CCI_direction'] = (panel_clean['CCI_next'] > panel_clean['CCI']).astype(int)

mask = panel_clean['CCI_next'].notna()
X_log = panel_clean.loc[mask, features_for_lr].values
y_log = panel_clean.loc[mask, 'CCI_direction'].values

scaler_log = StandardScaler()
X_log_s = scaler_log.fit_transform(X_log)

logreg = LogisticRegression(max_iter=1000, random_state=42)
cv_acc = cross_val_score(logreg, X_log_s, y_log, cv=kf, scoring='accuracy')
cv_auc = cross_val_score(logreg, X_log_s, y_log, cv=kf, scoring='roc_auc')

print(f"=== Logistic Regression — CCI yön tahmini ===")
print(f"Class dağılımı: {np.bincount(y_log)}")
print(f"CV Accuracy: {cv_acc.mean():.3f} ± {cv_acc.std():.3f}")
print(f"CV AUC:      {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

logreg.fit(X_log_s, y_log)
y_pred = logreg.predict(X_log_s)
cm = confusion_matrix(y_log, y_pred)

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Düşüş','Artış'], yticklabels=['Düşüş','Artış'])
ax.set_xlabel('Tahmin'); ax.set_ylabel('Gerçek')
ax.set_title(f'Confusion Matrix — CCI Direction (AUC={cv_auc.mean():.2f})')
plt.tight_layout()
plt.savefig('figures/ml_04_logreg_confusion.png', dpi=150)
plt.show()


In [ ]:
rf_features = WEATHER_FEATURES + ['unemployment_rate','inflation_log','covid_dummy',
                                    'month_sin','month_cos']
X_rf = panel_clean[rf_features].values
y_rf = panel_clean['CCI'].values

rf = RandomForestRegressor(n_estimators=200, max_depth=8, 
                            random_state=42, n_jobs=-1)
cv_rmse_rf = -cross_val_score(rf, X_rf, y_rf, cv=kf, scoring='neg_root_mean_squared_error')
cv_r2_rf = cross_val_score(rf, X_rf, y_rf, cv=kf, scoring='r2')

print(f"=== Random Forest — CCI ===")
print(f"CV RMSE: {cv_rmse_rf.mean():.3f}")
print(f"CV R²:   {cv_r2_rf.mean():.3f}")

rf.fit(X_rf, y_rf)
imp_df = pd.DataFrame({
    'feature': rf_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10,7))
ax.barh(imp_df['feature'], imp_df['importance'], color='forestgreen')
ax.set_title(f'Random Forest Feature Importance — CCI (R²={cv_r2_rf.mean():.2f})')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('figures/ml_05_rf_importance_cci.png', dpi=150)
plt.show()
print(imp_df.iloc[::-1])

In [ ]:
# Retail için RF
X_rf_r = panel_retail[rf_features].values
y_rf_r = panel_retail['retail_index'].values

rf_r = RandomForestRegressor(n_estimators=200, max_depth=8, 
                              random_state=42, n_jobs=-1)
cv_rmse_rf_r = -cross_val_score(rf_r, X_rf_r, y_rf_r, cv=kf, scoring='neg_root_mean_squared_error')
cv_r2_rf_r = cross_val_score(rf_r, X_rf_r, y_rf_r, cv=kf, scoring='r2')
print(f"=== Random Forest — Retail ===")
print(f"CV RMSE: {cv_rmse_rf_r.mean():.3f}, R²: {cv_r2_rf_r.mean():.3f}")

rf_r.fit(X_rf_r, y_rf_r)
imp_df_r = pd.DataFrame({
    'feature': rf_features,
    'importance': rf_r.feature_importances_
}).sort_values('importance', ascending=True)

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
cv_rmse_gb = -cross_val_score(gb, X_rf, y_rf, cv=kf, scoring='neg_root_mean_squared_error')
cv_r2_gb = cross_val_score(gb, X_rf, y_rf, cv=kf, scoring='r2')
print(f"\n=== Gradient Boosting — CCI ===")
print(f"CV RMSE: {cv_rmse_gb.mean():.3f}, R²: {cv_r2_gb.mean():.3f}")

# Retail RF importance grafik
fig, ax = plt.subplots(figsize=(10,7))
ax.barh(imp_df_r['feature'], imp_df_r['importance'], color='darkorange')
ax.set_title(f'RF Feature Importance — Retail (R²={cv_r2_rf_r.mean():.2f})')
plt.tight_layout()
plt.savefig('figures/ml_06_rf_importance_retail.png', dpi=150)
plt.show()


In [ ]:
# ============================================================
# KNN — Hafta 8 (z-score standardizasyonu zorunlu)
# K değeri grid search ile seçiliyor
# ============================================================
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# --- KNN Regresyon (CCI) ---
X_knn = panel_clean[rf_features].values
y_knn = panel_clean['CCI'].values

knn_pipe = Pipeline([
    ('scaler', StandardScaler()),   # z-score zorunlu
    ('knn', KNeighborsRegressor())
])
knn_grid = GridSearchCV(
    knn_pipe, 
    param_grid={'knn__n_neighbors': [3, 5, 7, 11, 15, 21]},
    cv=kf, scoring='r2', n_jobs=-1
)
knn_grid.fit(X_knn, y_knn)

best_k = knn_grid.best_params_['knn__n_neighbors']
cv_r2_knn = knn_grid.best_score_

# RMSE ayrı al (negatif scoring ile)
knn_rmse = -cross_val_score(knn_grid.best_estimator_, X_knn, y_knn, 
                              cv=kf, scoring='neg_root_mean_squared_error')
cv_rmse_knn = knn_rmse

print(f"=== KNN — CCI ===")
print(f"En iyi K: {best_k}")
print(f"CV R²: {cv_r2_knn:.3f}")
print(f"CV RMSE: {cv_rmse_knn.mean():.3f}")

# --- KNN Regresyon (Retail) ---
X_knn_r = panel_retail[rf_features].values
y_knn_r = panel_retail['retail_index'].values

knn_grid_r = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor())]),
    param_grid={'knn__n_neighbors': [3, 5, 7, 11, 15, 21]},
    cv=kf, scoring='r2', n_jobs=-1
)
knn_grid_r.fit(X_knn_r, y_knn_r)

best_k_r = knn_grid_r.best_params_['knn__n_neighbors']
cv_r2_knn_r = knn_grid_r.best_score_
cv_rmse_knn_r = -cross_val_score(knn_grid_r.best_estimator_, X_knn_r, y_knn_r,
                                   cv=kf, scoring='neg_root_mean_squared_error')

print(f"\n=== KNN — Retail ===")
print(f"En iyi K: {best_k_r}")
print(f"CV R²: {cv_r2_knn_r:.3f}")
print(f"CV RMSE: {cv_rmse_knn_r.mean():.3f}")

# K seçimi grafik
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(knn_grid.param_grid['knn__n_neighbors'],
             knn_grid.cv_results_['mean_test_score'], 'o-', color='steelblue')
axes[0].axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
axes[0].set_xlabel('K'); axes[0].set_ylabel('CV R²')
axes[0].set_title('KNN — K Seçimi (CCI)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(knn_grid_r.param_grid['knn__n_neighbors'],
             knn_grid_r.cv_results_['mean_test_score'], 'o-', color='darkorange')
axes[1].axvline(best_k_r, color='red', linestyle='--', label=f'Best K={best_k_r}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('CV R²')
axes[1].set_title('KNN — K Seçimi (Retail)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/ml_11_knn_k_selection.png', dpi=150)
plt.show()


In [ ]:
# ============================================================
# Tek Karar Ağacı — Hafta 8 (Gini impurity, görselleştirilebilir kurallar)
# ============================================================
from sklearn.tree import DecisionTreeRegressor, plot_tree

# Sığ ağaç ki okunabilir olsun
dtree = DecisionTreeRegressor(max_depth=4, min_samples_split=20, 
                                criterion='squared_error', random_state=42)
cv_rmse_dt = -cross_val_score(dtree, X_rf, y_rf, cv=kf, scoring='neg_root_mean_squared_error')
cv_r2_dt = cross_val_score(dtree, X_rf, y_rf, cv=kf, scoring='r2')
print(f"=== Decision Tree (max_depth=4) — CCI ===")
print(f"CV RMSE: {cv_rmse_dt.mean():.3f}")
print(f"CV R²: {cv_r2_dt.mean():.3f}")

# Görselleştir
dtree.fit(X_rf, y_rf)
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dtree, feature_names=rf_features, filled=True, rounded=True, 
          fontsize=9, ax=ax, impurity=True)
ax.set_title('Karar Ağacı — CCI Tahmini (max_depth=4)', fontsize=13)
plt.tight_layout()
plt.savefig('figures/ml_12_decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

# En önemli ilk split kuralları
print("\nİlk 5 önemli özellik (Decision Tree):")
imp_dt = pd.DataFrame({'feature': rf_features, 
                        'importance': dtree.feature_importances_}
                       ).sort_values('importance', ascending=False)
print(imp_dt.head().to_string(index=False))


In [ ]:
# ============================================================
# Grid Search ile Random Forest Hiperparametre Optimizasyonu (Hafta 8)
# ============================================================
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid={
        'n_estimators':     [100, 200, 300],
        'max_depth':        [5, 8, 12, None],
        'min_samples_split':[2, 5, 10],
    },
    cv=kf, scoring='r2', n_jobs=-1, verbose=0
)
rf_grid.fit(X_rf, y_rf)

print(f"=== RF Grid Search — CCI ===")
print(f"En iyi parametreler: {rf_grid.best_params_}")
print(f"En iyi CV R²: {rf_grid.best_score_:.3f}")

# Top 5 kombinasyon
results_df = pd.DataFrame(rf_grid.cv_results_)
top5 = results_df[['param_n_estimators','param_max_depth',
                    'param_min_samples_split','mean_test_score']].nlargest(5, 'mean_test_score')
print("\nEn iyi 5 kombinasyon:")
print(top5.to_string(index=False))


In [ ]:
# Hava değişkenlerine göre 480 country-month'u kümele
X_cluster = scaler.fit_transform(panel_clean[WEATHER_FEATURES])

# Optimal k (Elbow + Silhouette)
from sklearn.metrics import silhouette_score
inertias = []
sils = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_cluster, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(2,9), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[1].plot(range(2,9), sils, 'o-', color='darkorange')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score')
plt.tight_layout()
plt.savefig('figures/ml_07_kmeans_choice.png', dpi=150)
plt.show()

# k=4 ile fit
km = KMeans(n_clusters=4, random_state=42, n_init=10)
panel_clean['cluster'] = km.fit_predict(X_cluster)

# Her küme için ortalama CCI ve retail
cluster_summary = panel_clean.groupby('cluster').agg({
    'sunshine_hours':'mean',
    'temperature_2m_mean':'mean',
    'precipitation_sum':'mean',
    'CCI':'mean',
    'retail_index':'mean',
    'country':lambda x: x.mode()[0]
}).round(2)
print("Küme özetleri:")
print(cluster_summary)

# Görselleştir: cluster × country dağılımı
fig, ax = plt.subplots(figsize=(10,5))
ct = pd.crosstab(panel_clean['country'], panel_clean['cluster'])
ct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_title('Country-Month Cluster Dağılımı')

ax.set_ylabel('Ay sayısı')
plt.tight_layout()
plt.savefig('figures/ml_08_cluster_distribution.png', dpi=150)
plt.show()


In [ ]:
# ============================================================
# Hierarchical Clustering — 4 linkage yöntemi karşılaştırması (Hafta 9)
# ============================================================
country_profile = panel_clean.groupby('country')[WEATHER_FEATURES].mean()
country_scaled = StandardScaler().fit_transform(country_profile)

linkage_methods = ['single', 'complete', 'average', 'ward']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, method in zip(axes.flat, linkage_methods):
    Z = linkage(country_scaled, method=method)
    dendrogram(Z, labels=country_profile.index.tolist(), ax=ax)
    ax.set_title(f'Linkage: {method.capitalize()}', fontsize=12)
    ax.set_ylabel('Mesafe')

plt.suptitle('Hierarchical Clustering — Linkage Yöntemi Karşılaştırması', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/ml_13_hierarchical_linkage_compare.png', dpi=150)
plt.show()

print("Linkage yöntemleri arasındaki fark:")
print("- Single: en yakın iki noktaya göre — zincirleme etkisi olabilir")
print("- Complete: en uzak iki noktaya göre — kompakt kümeler")
print("- Average: ortalama mesafe — dengeli")
print("- Ward: küme içi varyansı minimize eder — istatistiksel olarak en sağlam")


In [ ]:
# Defansif final tablo — tüm modeller dahil
def _get(name):
    val = globals().get(name)
    if val is None:
        return None
    try:
        return val.mean()
    except AttributeError:
        return val

models_to_report = [
    ('Linear Regression',   'CCI',           'cv_rmse',     'cv_r2'),
    ('Linear Regression',   'Retail',        'cv_rmse_r',   'cv_r2_r'),
    ('Logistic Regression', 'CCI direction', None,          'cv_auc'),
    ('Decision Tree',       'CCI',           'cv_rmse_dt',  'cv_r2_dt'),
    ('KNN',                 'CCI',           'cv_rmse_knn', 'cv_r2_knn'),
    ('KNN',                 'Retail',        'cv_rmse_knn_r','cv_r2_knn_r'),
    ('Random Forest',       'CCI',           'cv_rmse_rf',  'cv_r2_rf'),
    ('Random Forest',       'Retail',        'cv_rmse_rf_r','cv_r2_rf_r'),
    ('Gradient Boosting',   'CCI',           'cv_rmse_gb',  'cv_r2_gb'),
]

rows, missing = [], []
for model, target, rmse_var, r2_var in models_to_report:
    rmse_val = _get(rmse_var) if rmse_var else np.nan
    r2_val   = _get(r2_var)
    if r2_val is None:
        missing.append(f"{model} ({target})")
        continue
    rows.append({
        'model': model, 'target': target,
        'RMSE': rmse_val if rmse_val is not None else np.nan,
        'R²': r2_val
    })

if missing:
    print("⚠️  Atlandı:", ", ".join(missing), "\n")

results = pd.DataFrame(rows)
print("=== FINAL — Tüm Modeller ===")
print(results.round(3).to_string(index=False))
results.to_csv('data/model_comparison.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_data = results.dropna(subset=['RMSE'])
sns.barplot(data=plot_data, x='model', y='RMSE', hue='target', ax=axes[0])
axes[0].set_title('RMSE Karşılaştırması (düşük = iyi)')
axes[0].tick_params(axis='x', rotation=35)

sns.barplot(data=results, x='model', y='R²', hue='target', ax=axes[1])
axes[1].set_title('R² Karşılaştırması (yüksek = iyi)')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig('figures/ml_10_model_comparison.png', dpi=150)
plt.show()
